# Stormlight BERTopic topic modeling

Fits one BERTopic model on every paragraph from *The Way of Kings* and *Words of Radiance* combined, then drives the standard visualization menu:

| Cell | View | What it answers |
|---|---|---|
| 4 | Topic inventory + barchart | What topics did the model find? |
| 5 | Intertopic distance map | Which topics are similar / surprising neighbors? |
| 6 | Topics per POV | Which topics belong to Kaladin vs. Shallan vs. Dalinar? |
| 7 | Topics over time | How does topic prominence shift through the narrative? |
| 8 | Topic hierarchy | How do topics cluster into themes? |
| 9 | Find topics by query | Hunt specific themes (Shardblades, highstorms, etc.) |

The trained model is cached to `bertopic_model/` — delete it to force a re-fit.


In [ ]:
# ── Cell 2: Imports + config ─────────────────────────────────────────────────
import re
from pathlib import Path
import numpy as np
import pandas as pd
from bertopic import BERTopic

REPO         = Path("/Users/caputomachine/Desktop/StormlightCorpus")
PARAS_IN     = REPO / "csv_data" / "paragraphs.csv"
MODEL_CACHE  = REPO / "stormlight_viz" / "bertopic_model"

MIN_WORDS_FILTER = 5                       # drop paragraphs shorter than this
EMBED_MODEL      = "all-MiniLM-L6-v2"      # fast, good-quality default

# Output HTML names
HTML_BARCHART       = "bertopic_barchart.html"
HTML_TOPICS_MAP     = "bertopic_intertopic_map.html"
HTML_TOPICS_PER_POV = "bertopic_topics_per_pov.html"
HTML_TOPICS_OVERTIME= "bertopic_topics_over_time.html"
HTML_HIERARCHY      = "bertopic_hierarchy.html"


In [ ]:
# ── Cell 3: Load paragraphs, filter, light cleanup ───────────────────────────
def clean_for_topics(t: str) -> str:
    if not isinstance(t, str): return ""
    t = re.sub(r"\*+", "", t)        # strip italic/bold markers
    t = re.sub(r"-{2,}", " ", t)      # collapse --- to a space
    t = re.sub(r"\s+", " ", t).strip()
    return t

paras = pd.read_csv(PARAS_IN)
print(f"raw paragraphs   : {len(paras):,}")

paras = paras[paras["n_words"] >= MIN_WORDS_FILTER].reset_index(drop=True)
paras["doc"] = paras["text"].map(clean_for_topics)
paras = paras[paras["doc"].str.len() > 0].reset_index(drop=True)
print(f"after filter (≥{MIN_WORDS_FILTER} words): {len(paras):,}")

docs = paras["doc"].tolist()
print(f"\nsample paragraphs:")
for i in (0, 1000, 10000):
    print(f"  [{i}] {docs[i][:120]}...")


In [ ]:
# ── Build CountVectorizer with Stormlight stopwords ──────────────────────────
# Loads `stormlight_stopwords.txt` (character/place names + literary noise) and
# unions with sklearn's English stopword list. The CountVectorizer drives the
# c-TF-IDF representation; edit the .txt file and re-run the next cell to
# refresh topic words WITHOUT retraining.
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

STOPWORDS_FILE = REPO / "stormlight_viz" / "stormlight_stopwords.txt"

def load_stopwords_file(path: Path) -> set[str]:
    words = set()
    with open(path) as f:
        for line in f:
            line = line.split("#", 1)[0].strip().lower()
            if line:
                words.add(line)
    return words

extra_sw    = load_stopwords_file(STOPWORDS_FILE)
all_sw      = sorted(set(ENGLISH_STOP_WORDS) | extra_sw)
print(f"{len(extra_sw)} Stormlight stopwords + {len(ENGLISH_STOP_WORDS)} English stopwords = {len(all_sw)} total")

vectorizer_model = CountVectorizer(
    stop_words=all_sw,
    min_df=10,            # word must appear in ≥10 paragraphs to be considered
    ngram_range=(1, 1),   # switch to (1, 2) to include bigrams like "knights radiant"
)


In [ ]:
# ── Train BERTopic (or load cached) + apply stopword filter ───────────────────
# First run embeds 28k paragraphs with `all-MiniLM-L6-v2` — takes ~3–5 minutes
# on Apple-silicon CPU. The trained model + embeddings are cached for instant
# reloads. Delete `bertopic_model/` to force a re-fit.

if MODEL_CACHE.exists():
    print(f"loading cached model from {MODEL_CACHE.name}/")
    topic_model = BERTopic.load(str(MODEL_CACHE), embedding_model=EMBED_MODEL)
    topics = topic_model.topics_
    print(f"loaded; {len(set(topics))} topics (incl. outliers)")
    # Re-extract topic words using the current vectorizer (stopword filter).
    # This is fast — only c-TF-IDF is recomputed; clustering is unchanged.
    print("re-extracting topic words with stopword filter ...")
    topic_model.update_topics(docs, vectorizer_model=vectorizer_model)
    print("done")
else:
    print(f"training BERTopic on {len(docs):,} paragraphs ...")
    topic_model = BERTopic(
        embedding_model=EMBED_MODEL,
        vectorizer_model=vectorizer_model,
        nr_topics="auto",
        top_n_words=15,
        verbose=True,
        calculate_probabilities=False,
    )
    topics, _ = topic_model.fit_transform(docs)
    MODEL_CACHE.mkdir(parents=True, exist_ok=True)
    topic_model.save(str(MODEL_CACHE), serialization="safetensors",
                     save_ctfidf=True, save_embedding_model=EMBED_MODEL)
    print(f"saved to {MODEL_CACHE.name}/")

# Attach topic labels to the metadata frame for downstream cells
paras["topic"] = topics
n_topics = len(set(topics)) - (1 if -1 in topics else 0)
print(f"\n{n_topics} non-outlier topics  |  {(paras['topic']==-1).sum():,} outlier paragraphs")


In [ ]:
# ── Cell 5: Topic inventory + barchart of top words ──────────────────────────
info = topic_model.get_topic_info()
print(f"{len(info)} topics total (incl. outlier topic -1)")
print()
print(info.head(20).to_string(index=False))

fig = topic_model.visualize_barchart(top_n_topics=16, n_words=10, height=240)
fig.write_html(HTML_BARCHART, include_plotlyjs="cdn")
print(f"\nsaved {HTML_BARCHART}")
fig.show()


In [ ]:
# ── Cell 6: Intertopic distance map ──────────────────────────────────────────
fig = topic_model.visualize_topics()
fig.write_html(HTML_TOPICS_MAP, include_plotlyjs="cdn")
print(f"saved {HTML_TOPICS_MAP}")
fig.show()


In [ ]:
# ── Cell 7: Topics per POV character ─────────────────────────────────────────
# The headline plot for a multi-POV corpus: which topics belong to which POV.

# Treat multi-POV "Dalinar/Adolin" as its own class. Keep top 8 by paragraph count.
povs_series = paras["pov"].fillna("Unknown")
top_pov_classes = povs_series.value_counts().head(8).index.tolist()
povs = [p if p in top_pov_classes else "Other" for p in povs_series]

topics_per_class = topic_model.topics_per_class(docs, classes=povs)
fig = topic_model.visualize_topics_per_class(
    topics_per_class, top_n_topics=20, height=900,
)
fig.write_html(HTML_TOPICS_PER_POV, include_plotlyjs="cdn")
print(f"saved {HTML_TOPICS_PER_POV}")
print("\nClass coverage:")
print(pd.Series(povs).value_counts().to_string())
fig.show()


In [ ]:
# ── Cell 8: Topics over time (narrative position) ────────────────────────────
# Use a synthetic timestamp = (book_index * 1000) + chapter_order so the timeline
# runs WoK first then WoR, monotonically.
book_offset = {"The Way of Kings": 0, "Words of Radiance": 1000}
timestamps = paras.apply(lambda r: book_offset[r["book"]] + int(r["chapter_order"]), axis=1).tolist()

topics_over_time = topic_model.topics_over_time(docs, timestamps, nr_bins=40)

fig = topic_model.visualize_topics_over_time(
    topics_over_time, top_n_topics=12, height=550,
)
# Mark the book boundary
fig.add_vline(x=1000, line_dash="dash", line_color="grey",
              annotation_text="WoR begins", annotation_position="top right")
fig.write_html(HTML_TOPICS_OVERTIME, include_plotlyjs="cdn")
print(f"saved {HTML_TOPICS_OVERTIME}")
fig.show()


In [ ]:
# ── Cell 9: Hierarchical topic structure ─────────────────────────────────────
fig = topic_model.visualize_hierarchy(top_n_topics=40)
fig.write_html(HTML_HIERARCHY, include_plotlyjs="cdn")
print(f"saved {HTML_HIERARCHY}")
fig.show()


In [ ]:
# ── Cell 10: Search topics by keyword ────────────────────────────────────────
# Quick way to interrogate the topic space.

queries = [
    "Shardblade Shardplate weapons",
    "highstorm wind lightning rain",
    "Bridge Four soldiers spears",
    "Shallan drawing sketching memory",
    "spren bond honor oaths",
    "Parshendi listener songs",
]

for q in queries:
    sim_topics, sim_scores = topic_model.find_topics(q, top_n=3)
    print(f"\n[{q!r}]")
    for t, s in zip(sim_topics, sim_scores):
        words = ", ".join(w for w, _ in topic_model.get_topic(t)[:8])
        size  = (paras["topic"] == t).sum()
        print(f"  topic {t:>3} (n={size:>4}, sim={s:.2f}): {words}")
